# AG03.5: Function Calling Deep Dive

**Duration:** 8 hours  
**Prerequisites:** AG03 (Tools & Function Calling)

---

## Learning Objectives

By the end of this lesson, you will be able to:

1. Design robust tool schemas with complex parameter types
2. Implement comprehensive error handling for function calls
3. Handle parallel function calling efficiently
4. Debug common function calling failures
5. Build production-ready tools with validation and retry logic

---

## Why Function Calling Matters

Function calling is the bridge between LLMs and the real world. Poor function design leads to:

- **Hallucinated parameters** - LLM invents values that don't exist
- **Silent failures** - Tool errors that the agent can't recover from
- **Inefficient execution** - Sequential calls when parallel would work
- **Poor UX** - Long waits for results, unclear error messages

This lesson teaches you to build tools that LLMs can reliably use in production.

---

## Part 1: Tool Schema Design Patterns

### 1.1 Basic Schema Review

In [ ]:
import sys
sys.path.append('..')

from agentkit import tool, run_agent
import json
from typing import List, Dict, Optional, Literal, Union
from pydantic import BaseModel, Field, validator
from datetime import datetime, date
import time

# Basic tool - what you learned in AG03
@tool
def search_products(query: str, max_results: int = 5) -> str:
    """Search for products in the catalog.
    
    Args:
        query: Search keywords
        max_results: Maximum number of results to return
    """
    # Simulated search
    products = [
        {"id": 1, "name": "Laptop", "price": 999},
        {"id": 2, "name": "Mouse", "price": 25},
        {"id": 3, "name": "Keyboard", "price": 75},
    ]
    return json.dumps(products[:max_results])

print("✓ Basic tool defined")

### 1.2 Complex Parameter Types

Real-world tools need more than strings and integers. Let's explore advanced parameter patterns.

In [ ]:
# Pattern 1: Enums for constrained choices
@tool
def filter_products(
    category: Literal["electronics", "clothing", "books", "home"],
    sort_by: Literal["price_asc", "price_desc", "rating", "newest"] = "rating",
    min_price: Optional[float] = None,
    max_price: Optional[float] = None
) -> str:
    """Filter products by category and price range.
    
    Args:
        category: Product category to filter by
        sort_by: How to sort results
        min_price: Minimum price (optional)
        max_price: Maximum price (optional)
    """
    filters = {
        "category": category,
        "sort": sort_by,
        "price_range": [min_price, max_price]
    }
    return f"Filtered products: {json.dumps(filters)}"

print("✓ Enum-based tool defined")

In [ ]:
# Pattern 2: Complex objects with Pydantic
class DateRange(BaseModel):
    start: date = Field(description="Start date in YYYY-MM-DD format")
    end: date = Field(description="End date in YYYY-MM-DD format")
    
    @validator('end')
    def end_after_start(cls, v, values):
        if 'start' in values and v < values['start']:
            raise ValueError('End date must be after start date')
        return v

class OrderFilter(BaseModel):
    status: List[Literal["pending", "shipped", "delivered", "cancelled"]]
    date_range: Optional[DateRange] = None
    customer_id: Optional[int] = None
    min_amount: Optional[float] = Field(None, ge=0, description="Minimum order amount")

@tool
def search_orders(filters: OrderFilter) -> str:
    """Search orders with complex filtering.
    
    Args:
        filters: Order filter criteria including status, date range, customer, and amount
    """
    # Pydantic automatically validates the structure
    return f"Orders matching: {filters.model_dump_json()}"

print("✓ Pydantic-based tool defined")

In [ ]:
# Pattern 3: Arrays of objects
class LineItem(BaseModel):
    product_id: int = Field(description="Product ID from catalog")
    quantity: int = Field(ge=1, le=100, description="Quantity to order (1-100)")
    notes: Optional[str] = Field(None, max_length=200)

@tool
def create_order(
    customer_id: int,
    items: List[LineItem],
    shipping_address: str,
    express_shipping: bool = False
) -> str:
    """Create a new order with multiple items.
    
    Args:
        customer_id: Customer ID placing the order
        items: List of items to order (product_id, quantity, optional notes)
        shipping_address: Full shipping address
        express_shipping: Whether to use express shipping (default: False)
    """
    total_items = sum(item.quantity for item in items)
    order = {
        "customer_id": customer_id,
        "items": [item.model_dump() for item in items],
        "total_items": total_items,
        "shipping": "express" if express_shipping else "standard",
        "address": shipping_address
    }
    return f"Order created: {json.dumps(order, indent=2)}"

print("✓ Array-based tool defined")

**Exercise 1.2:** Test these tools with an agent to see how the LLM handles complex parameters.

In [ ]:
# Test complex parameter handling
response = run_agent(
    prompt="Find electronics sorted by price (low to high) under $500, then create an order for customer 12345 with 2 laptops and 1 mouse, ship to '123 Main St, San Francisco, CA 94105', use express shipping",
    tools=[filter_products, create_order],
    model="gpt-4o-mini"
)

print("\n=== Agent Response ===")
print(response)

### 1.3 Schema Design Best Practices

**DO:**
- Use `Literal` types for fixed choices (helps LLM pick valid values)
- Provide detailed descriptions for every parameter
- Use Pydantic validators for business logic constraints
- Set reasonable defaults for optional parameters
- Use `Field()` with `ge`, `le`, `max_length` for validation

**DON'T:**
- Use ambiguous parameter names (`data`, `info`, `params`)
- Accept arbitrary JSON blobs without validation
- Make everything required (LLM struggles with many required params)
- Use overly complex nested structures (>3 levels deep)

---

## Part 2: Error Handling & Validation

### 2.1 Tool-Level Error Handling

In [ ]:
from typing import Dict, Any
import traceback

class ToolError(Exception):
    """Base class for tool errors that the agent can understand."""
    pass

class ValidationError(ToolError):
    """Parameter validation failed."""
    pass

class ResourceNotFoundError(ToolError):
    """Requested resource doesn't exist."""
    pass

class PermissionError(ToolError):
    """User doesn't have permission for this action."""
    pass

def safe_tool_execution(func):
    """Decorator to wrap tool execution with error handling."""
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            return {
                "success": True,
                "data": result
            }
        except ToolError as e:
            # Expected errors that agent can handle
            return {
                "success": False,
                "error": str(e),
                "error_type": type(e).__name__,
                "recoverable": True
            }
        except Exception as e:
            # Unexpected errors
            return {
                "success": False,
                "error": f"Internal error: {str(e)}",
                "error_type": "InternalError",
                "recoverable": False
            }
    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    return wrapper

print("✓ Error handling utilities defined")

In [ ]:
# Example: Database query tool with error handling
@tool
@safe_tool_execution
def get_customer(customer_id: int) -> Dict[str, Any]:
    """Retrieve customer information by ID.
    
    Args:
        customer_id: Customer ID to look up
        
    Returns:
        Customer data including name, email, and order history
        
    Raises:
        ValidationError: If customer_id is invalid
        ResourceNotFoundError: If customer doesn't exist
    """
    if customer_id <= 0:
        raise ValidationError(f"customer_id must be positive, got {customer_id}")
    
    # Simulated database lookup
    customers_db = {
        12345: {"name": "Alice", "email": "alice@example.com", "orders": 5},
        67890: {"name": "Bob", "email": "bob@example.com", "orders": 12}
    }
    
    if customer_id not in customers_db:
        raise ResourceNotFoundError(f"Customer {customer_id} not found. Try customer 12345 or 67890.")
    
    return customers_db[customer_id]

# Test error handling
print("Valid customer:", get_customer(12345))
print("\nInvalid ID:", get_customer(-1))
print("\nNot found:", get_customer(99999))

### 2.2 Retry Logic

Some failures are transient (network issues, rate limits). Build retry logic into tools that call external APIs.

In [ ]:
import time
from functools import wraps

def retry_with_backoff(max_retries: int = 3, base_delay: float = 1.0):
    """Retry decorator with exponential backoff."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    if attempt < max_retries - 1:
                        delay = base_delay * (2 ** attempt)
                        print(f"Attempt {attempt + 1} failed, retrying in {delay}s...")
                        time.sleep(delay)
                    else:
                        # Last attempt failed
                        raise last_exception
            
            raise last_exception
        return wrapper
    return decorator

@tool
@retry_with_backoff(max_retries=3, base_delay=0.5)
def call_external_api(endpoint: str) -> str:
    """Call an external API endpoint (simulated).
    
    Args:
        endpoint: API endpoint to call
    """
    import random
    
    # Simulate 70% failure rate on first try
    if random.random() < 0.7:
        raise ConnectionError("Network timeout")
    
    return f"Success: Data from {endpoint}"

# Test retry logic
print(call_external_api("/api/products"))

### 2.3 Validation at the Schema Level

Use Pydantic validators to catch bad data before tool execution.

In [ ]:
from pydantic import BaseModel, Field, validator, root_validator
import re

class CreateUserRequest(BaseModel):
    email: str = Field(description="User's email address")
    password: str = Field(description="Password (min 8 chars, must include number)")
    age: int = Field(ge=13, le=120, description="User's age (13-120)")
    username: str = Field(min_length=3, max_length=20, description="Username (3-20 chars)")
    
    @validator('email')
    def validate_email(cls, v):
        if not re.match(r'^[\w\.-]+@[\w\.-]+\.\w+$', v):
            raise ValueError('Invalid email format')
        return v.lower()
    
    @validator('password')
    def validate_password(cls, v):
        if len(v) < 8:
            raise ValueError('Password must be at least 8 characters')
        if not any(c.isdigit() for c in v):
            raise ValueError('Password must contain at least one number')
        return v
    
    @validator('username')
    def validate_username(cls, v):
        if not re.match(r'^[a-zA-Z0-9_]+$', v):
            raise ValueError('Username can only contain letters, numbers, and underscores')
        return v

@tool
def create_user(user: CreateUserRequest) -> str:
    """Create a new user account with validation.
    
    Args:
        user: User data (email, password, age, username)
    """
    # If we get here, all validation passed
    return f"User created: {user.username} ({user.email})"

# Test validation
try:
    response = run_agent(
        prompt="Create a user with email 'bad-email', password 'short', age 5, username 'ab'",
        tools=[create_user],
        model="gpt-4o-mini"
    )
    print(response)
except Exception as e:
    print(f"Validation failed (expected): {e}")

**Exercise 2.3:** Build a tool that accepts credit card information with proper validation (card number format, expiry date, CVV).

---

## Part 3: Parallel Function Calling

### 3.1 Understanding Parallel Execution

Modern LLMs (GPT-4, Claude 3+) can call multiple functions in a single response. This dramatically improves efficiency.

In [ ]:
import time

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city.
    
    Args:
        city: City name
    """
    time.sleep(1)  # Simulate API call
    return f"Weather in {city}: 72°F, Sunny"

@tool
def get_population(city: str) -> str:
    """Get population of a city.
    
    Args:
        city: City name
    """
    time.sleep(1)  # Simulate API call
    populations = {
        "San Francisco": "873,965",
        "New York": "8,336,817",
        "Austin": "978,908"
    }
    return f"Population of {city}: {populations.get(city, 'Unknown')}"

@tool
def get_timezone(city: str) -> str:
    """Get timezone of a city.
    
    Args:
        city: City name
    """
    time.sleep(1)  # Simulate API call
    timezones = {
        "San Francisco": "PST (UTC-8)",
        "New York": "EST (UTC-5)",
        "Austin": "CST (UTC-6)"
    }
    return f"Timezone of {city}: {timezones.get(city, 'Unknown')}"

# Test parallel execution
start = time.time()
response = run_agent(
    prompt="Tell me the weather, population, and timezone of San Francisco",
    tools=[get_weather, get_population, get_timezone],
    model="gpt-4o"
)
elapsed = time.time() - start

print(f"\n=== Response (took {elapsed:.1f}s) ===")
print(response)
print(f"\nNote: If parallel execution worked, this took ~1s (not ~3s)")

### 3.2 When to Use Parallel Calling

**Use parallel calls when:**
- Operations are independent (no data dependency)
- All operations can succeed/fail independently
- You need to aggregate data from multiple sources

**Use sequential calls when:**
- Second call depends on first call's result
- Operations modify shared state
- Order matters (e.g., create before update)

### 3.3 Handling Partial Failures

In [ ]:
@tool
def get_stock_price(symbol: str) -> str:
    """Get current stock price.
    
    Args:
        symbol: Stock ticker symbol (e.g., AAPL, GOOGL)
    """
    prices = {
        "AAPL": "$175.43",
        "GOOGL": "$141.80",
        "MSFT": "$378.91"
    }
    
    if symbol not in prices:
        return f"Error: Unknown symbol {symbol}. Available: AAPL, GOOGL, MSFT"
    
    return f"{symbol}: {prices[symbol]}"

# Test with mix of valid and invalid symbols
response = run_agent(
    prompt="Get stock prices for AAPL, GOOGL, and INVALID_SYMBOL",
    tools=[get_stock_price],
    model="gpt-4o-mini"
)

print(response)
print("\nNote: Agent should handle partial failures gracefully")

**Exercise 3.3:** Build a multi-source search agent that queries 3 different data sources in parallel and aggregates results.

---

## Part 4: Debugging Function Calling

### 4.1 Common Failure Modes

In [ ]:
# Failure Mode 1: Hallucinated Parameters
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email message.
    
    Args:
        to: Recipient email address
        subject: Email subject line
        body: Email body text
    """
    # Problem: LLM might invent email addresses if not given context
    if "@" not in to:
        return f"Error: '{to}' is not a valid email address. Please provide a real email."
    return f"Email sent to {to}"

# Test: Agent has no email addresses in context
response = run_agent(
    prompt="Send an email to John about the meeting",
    tools=[send_email],
    model="gpt-4o-mini"
)
print("\nWithout context:")
print(response)

In [ ]:
# Fix: Provide context or require explicit lookup
@tool
def lookup_email(name: str) -> str:
    """Look up email address by name.
    
    Args:
        name: Person's name
    """
    directory = {
        "John": "john@example.com",
        "Alice": "alice@example.com",
        "Bob": "bob@example.com"
    }
    return directory.get(name, f"No email found for {name}")

response = run_agent(
    prompt="Send an email to John about the meeting. Look up his email first.",
    tools=[lookup_email, send_email],
    model="gpt-4o-mini"
)
print("\nWith lookup tool:")
print(response)

### 4.2 Debugging Techniques

In [ ]:
# Technique 1: Log all function calls
def logged_tool(func):
    """Decorator to log tool invocations."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"\n[TOOL CALL] {func.__name__}")
        print(f"  Args: {args}")
        print(f"  Kwargs: {kwargs}")
        
        start = time.time()
        try:
            result = func(*args, **kwargs)
            elapsed = time.time() - start
            print(f"  Result: {result[:100]}..." if len(str(result)) > 100 else f"  Result: {result}")
            print(f"  Duration: {elapsed:.3f}s")
            return result
        except Exception as e:
            elapsed = time.time() - start
            print(f"  Error: {e}")
            print(f"  Duration: {elapsed:.3f}s")
            raise
    
    return wrapper

@tool
@logged_tool
def complex_calculation(a: int, b: int, operation: Literal["add", "multiply"]) -> str:
    """Perform a calculation."""
    if operation == "add":
        return f"{a} + {b} = {a + b}"
    else:
        return f"{a} × {b} = {a * b}"

response = run_agent(
    prompt="Calculate 15 + 27 and then multiply 8 by 9",
    tools=[complex_calculation],
    model="gpt-4o-mini"
)
print("\n=== Final Response ===")
print(response)

### 4.3 Schema Validation Testing

In [ ]:
# Test tool schemas programmatically
from pydantic import ValidationError

class PaymentRequest(BaseModel):
    amount: float = Field(gt=0, description="Payment amount (must be positive)")
    currency: Literal["USD", "EUR", "GBP"] = "USD"
    card_number: str = Field(min_length=13, max_length=19)
    
    @validator('card_number')
    def validate_card(cls, v):
        if not v.isdigit():
            raise ValueError('Card number must contain only digits')
        return v

def test_payment_schema():
    """Test payment validation with various inputs."""
    test_cases = [
        # (input, should_pass, description)
        ({"amount": 100, "currency": "USD", "card_number": "4111111111111111"}, True, "Valid payment"),
        ({"amount": -50, "currency": "USD", "card_number": "4111111111111111"}, False, "Negative amount"),
        ({"amount": 100, "currency": "JPY", "card_number": "4111111111111111"}, False, "Invalid currency"),
        ({"amount": 100, "currency": "USD", "card_number": "123"}, False, "Card too short"),
        ({"amount": 100, "currency": "USD", "card_number": "abcd1111111111"}, False, "Card non-numeric"),
    ]
    
    print("\nSchema Validation Tests:")
    print("=" * 60)
    
    for test_input, should_pass, description in test_cases:
        try:
            PaymentRequest(**test_input)
            result = "✓ PASS" if should_pass else "✗ FAIL (should have failed)"
        except ValidationError as e:
            result = "✗ FAIL (unexpected)" if should_pass else "✓ PASS"
        
        print(f"{result} - {description}")

test_payment_schema()

---

## Part 5: Production-Ready Patterns

### 5.1 Rate Limiting

In [ ]:
from collections import deque
from datetime import datetime, timedelta

class RateLimiter:
    """Token bucket rate limiter."""
    
    def __init__(self, max_calls: int, time_window: float):
        """
        Args:
            max_calls: Maximum calls allowed in time window
            time_window: Time window in seconds
        """
        self.max_calls = max_calls
        self.time_window = time_window
        self.calls = deque()
    
    def allow_call(self) -> bool:
        """Check if call is allowed under rate limit."""
        now = datetime.now()
        
        # Remove calls outside time window
        cutoff = now - timedelta(seconds=self.time_window)
        while self.calls and self.calls[0] < cutoff:
            self.calls.popleft()
        
        # Check if under limit
        if len(self.calls) < self.max_calls:
            self.calls.append(now)
            return True
        return False

# Apply rate limiting to a tool
api_limiter = RateLimiter(max_calls=5, time_window=10)  # 5 calls per 10 seconds

@tool
def rate_limited_api_call(query: str) -> str:
    """Call external API with rate limiting.
    
    Args:
        query: Search query
    """
    if not api_limiter.allow_call():
        return "Error: Rate limit exceeded. Please wait 10 seconds and try again."
    
    return f"API result for: {query}"

# Test rate limiting
print("Testing rate limiter (5 calls per 10s):")
for i in range(7):
    result = rate_limited_api_call(f"query_{i}")
    print(f"  Call {i+1}: {result}")
    time.sleep(0.1)

### 5.2 Caching Tool Results

In [ ]:
from functools import lru_cache
import hashlib

class TTLCache:
    """Simple time-based cache."""
    
    def __init__(self, ttl_seconds: int = 300):
        self.ttl_seconds = ttl_seconds
        self.cache = {}
    
    def get(self, key: str):
        if key in self.cache:
            value, timestamp = self.cache[key]
            if datetime.now() - timestamp < timedelta(seconds=self.ttl_seconds):
                return value
            else:
                del self.cache[key]
        return None
    
    def set(self, key: str, value: Any):
        self.cache[key] = (value, datetime.now())

# Use caching for expensive operations
expensive_cache = TTLCache(ttl_seconds=60)

@tool
def expensive_computation(n: int) -> str:
    """Perform expensive computation (cached for 60s).
    
    Args:
        n: Input number
    """
    cache_key = f"compute_{n}"
    
    # Check cache
    cached = expensive_cache.get(cache_key)
    if cached is not None:
        return f"[CACHED] Result for {n}: {cached}"
    
    # Simulate expensive computation
    print(f"  Computing result for {n}...")
    time.sleep(2)
    result = n ** 2
    
    # Store in cache
    expensive_cache.set(cache_key, result)
    return f"Result for {n}: {result}"

# Test caching
print("First call (should take 2s):")
print(expensive_computation(42))

print("\nSecond call (should be instant):")
print(expensive_computation(42))

### 5.3 Observability & Metrics

In [ ]:
from dataclasses import dataclass, field
from typing import List
import statistics

@dataclass
class ToolMetrics:
    """Track metrics for a tool."""
    name: str
    call_count: int = 0
    success_count: int = 0
    error_count: int = 0
    durations: List[float] = field(default_factory=list)
    
    def record_call(self, duration: float, success: bool):
        self.call_count += 1
        self.durations.append(duration)
        if success:
            self.success_count += 1
        else:
            self.error_count += 1
    
    def summary(self) -> str:
        if not self.durations:
            return f"{self.name}: No calls yet"
        
        avg_duration = statistics.mean(self.durations)
        p95_duration = sorted(self.durations)[int(len(self.durations) * 0.95)]
        success_rate = (self.success_count / self.call_count) * 100
        
        return f"""
{self.name}:
  Calls: {self.call_count}
  Success Rate: {success_rate:.1f}%
  Avg Duration: {avg_duration:.3f}s
  P95 Duration: {p95_duration:.3f}s
"""

# Global metrics registry
metrics_registry = {}

def instrumented_tool(func):
    """Decorator to add metrics tracking."""
    tool_name = func.__name__
    metrics_registry[tool_name] = ToolMetrics(name=tool_name)
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        success = True
        
        try:
            result = func(*args, **kwargs)
            return result
        except Exception as e:
            success = False
            raise
        finally:
            duration = time.time() - start
            metrics_registry[tool_name].record_call(duration, success)
    
    return wrapper

@tool
@instrumented_tool
def monitored_search(query: str) -> str:
    """Search with metrics tracking."""
    time.sleep(0.1 + 0.05 * len(query) / 10)  # Variable duration
    return f"Results for: {query}"

# Run some queries
for query in ["short", "medium query", "very long search query here"] * 3:
    monitored_search(query)

# Print metrics
print("\n=== Tool Metrics ===")
for tool_name, metrics in metrics_registry.items():
    print(metrics.summary())

---

## Part 6: Advanced Patterns

### 6.1 Streaming Tool Results

In [ ]:
from typing import Iterator

@tool
def stream_large_dataset(query: str, page_size: int = 10) -> str:
    """Stream large dataset in chunks.
    
    Args:
        query: Dataset query
        page_size: Number of records per page
    """
    # Simulate large dataset
    total_records = 47
    
    results = []
    for page in range((total_records + page_size - 1) // page_size):
        start = page * page_size
        end = min(start + page_size, total_records)
        results.append(f"Page {page + 1}: Records {start}-{end}")
    
    return f"Dataset '{query}' streamed in {len(results)} pages:\n" + "\n".join(results)

print(stream_large_dataset("customer_data", page_size=15))

### 6.2 Tool Composition

Build complex tools from simpler ones.

In [ ]:
# Simple tools
@tool
def validate_product_id(product_id: int) -> str:
    """Validate product ID exists."""
    valid_ids = [101, 102, 103]
    if product_id not in valid_ids:
        raise ValidationError(f"Product {product_id} not found")
    return f"Product {product_id} is valid"

@tool
def check_inventory(product_id: int) -> str:
    """Check product inventory."""
    inventory = {101: 50, 102: 0, 103: 23}
    stock = inventory.get(product_id, 0)
    return f"Product {product_id}: {stock} units in stock"

@tool
def calculate_price(product_id: int, quantity: int) -> str:
    """Calculate total price with discounts."""
    prices = {101: 99.99, 102: 149.99, 103: 29.99}
    unit_price = prices.get(product_id, 0)
    
    # Volume discount
    discount = 0.1 if quantity >= 10 else 0
    total = unit_price * quantity * (1 - discount)
    
    return f"Total: ${total:.2f} ({quantity} × ${unit_price:.2f}, {int(discount*100)}% discount)"

# Composite tool
@tool
def purchase_product(product_id: int, quantity: int) -> str:
    """Complete purchase with validation, inventory check, and pricing.
    
    Args:
        product_id: Product ID to purchase
        quantity: Quantity to purchase
    """
    # Compose multiple tools
    try:
        # Step 1: Validate
        validate_product_id(product_id)
        
        # Step 2: Check inventory
        inventory_result = check_inventory(product_id)
        stock = int(inventory_result.split(':')[1].split()[0])
        
        if stock < quantity:
            return f"Error: Insufficient stock. Requested {quantity}, available {stock}"
        
        # Step 3: Calculate price
        price_result = calculate_price(product_id, quantity)
        
        return f"Purchase successful!\n{price_result}\nRemaining stock: {stock - quantity}"
        
    except ToolError as e:
        return f"Purchase failed: {e}"

# Test composite tool
print(purchase_product(101, 5))
print("\n" + purchase_product(102, 1))  # Out of stock
print("\n" + purchase_product(999, 1))  # Invalid product

### 6.3 Context-Aware Tools

Tools that adapt based on conversation context or user state.

In [ ]:
class UserContext:
    """Track user state across tool calls."""
    
    def __init__(self):
        self.user_id: Optional[int] = None
        self.shopping_cart: List[Dict] = []
        self.preferences: Dict = {}
    
    def is_authenticated(self) -> bool:
        return self.user_id is not None

# Global context (in production, use session management)
user_context = UserContext()

@tool
def login(user_id: int) -> str:
    """Log in as a user."""
    user_context.user_id = user_id
    return f"Logged in as user {user_id}"

@tool
def add_to_cart(product_id: int, quantity: int = 1) -> str:
    """Add item to shopping cart (requires authentication)."""
    if not user_context.is_authenticated():
        return "Error: Please log in first using the login tool"
    
    user_context.shopping_cart.append({"product_id": product_id, "quantity": quantity})
    return f"Added {quantity}x product {product_id} to cart. Cart size: {len(user_context.shopping_cart)}"

@tool
def view_cart() -> str:
    """View shopping cart contents."""
    if not user_context.is_authenticated():
        return "Error: Please log in first"
    
    if not user_context.shopping_cart:
        return "Your cart is empty"
    
    return f"Cart ({len(user_context.shopping_cart)} items): {json.dumps(user_context.shopping_cart)}"

# Test context-aware tools
print(add_to_cart(101, 2))  # Should fail - not logged in
print(login(12345))
print(add_to_cart(101, 2))  # Should succeed
print(add_to_cart(103, 1))
print(view_cart())

---

## Part 7: Real-World Case Study

Let's build a complete system: **E-commerce Order Management Agent**

In [ ]:
# Complete production-ready tool suite
from datetime import datetime
from typing import List, Optional
from pydantic import BaseModel, Field

# Models
class Product(BaseModel):
    id: int
    name: str
    price: float
    stock: int

class Order(BaseModel):
    id: int
    customer_id: int
    items: List[Dict[str, int]]
    total: float
    status: str
    created_at: datetime

# Database (simulated)
products_db = [
    Product(id=101, name="Laptop", price=999.99, stock=15),
    Product(id=102, name="Mouse", price=29.99, stock=50),
    Product(id=103, name="Keyboard", price=79.99, stock=30),
]

orders_db: List[Order] = []
order_id_counter = 1000

# Tools
@tool
@instrumented_tool
@logged_tool
def search_products_by_name(query: str) -> str:
    """Search products by name.
    
    Args:
        query: Product name to search for
    """
    results = [p for p in products_db if query.lower() in p.name.lower()]
    return json.dumps([p.model_dump() for p in results], indent=2)

@tool
@instrumented_tool
@logged_tool
def check_product_availability(product_id: int, quantity: int) -> str:
    """Check if product is available in requested quantity.
    
    Args:
        product_id: Product ID to check
        quantity: Desired quantity
    """
    product = next((p for p in products_db if p.id == product_id), None)
    if not product:
        return json.dumps({"available": False, "error": "Product not found"})
    
    available = product.stock >= quantity
    return json.dumps({
        "available": available,
        "product": product.name,
        "requested": quantity,
        "in_stock": product.stock
    })

@tool
@instrumented_tool
@logged_tool
def create_order_for_customer(
    customer_id: int,
    items: List[Dict[str, int]]
) -> str:
    """Create a new order.
    
    Args:
        customer_id: Customer ID
        items: List of items, each with 'product_id' and 'quantity'
    """
    global order_id_counter
    
    # Validate and calculate total
    total = 0.0
    for item in items:
        product = next((p for p in products_db if p.id == item['product_id']), None)
        if not product:
            return json.dumps({"success": False, "error": f"Product {item['product_id']} not found"})
        
        if product.stock < item['quantity']:
            return json.dumps({
                "success": False,
                "error": f"Insufficient stock for {product.name}: need {item['quantity']}, have {product.stock}"
            })
        
        total += product.price * item['quantity']
    
    # Create order
    order = Order(
        id=order_id_counter,
        customer_id=customer_id,
        items=items,
        total=total,
        status="pending",
        created_at=datetime.now()
    )
    orders_db.append(order)
    order_id_counter += 1
    
    # Update inventory
    for item in items:
        product = next(p for p in products_db if p.id == item['product_id'])
        product.stock -= item['quantity']
    
    return json.dumps({
        "success": True,
        "order_id": order.id,
        "total": total,
        "status": order.status
    })

@tool
@instrumented_tool
@logged_tool
def get_order_status(order_id: int) -> str:
    """Get status of an order.
    
    Args:
        order_id: Order ID to check
    """
    order = next((o for o in orders_db if o.id == order_id), None)
    if not order:
        return json.dumps({"error": "Order not found"})
    
    return order.model_dump_json()

print("✓ E-commerce tool suite ready")

In [ ]:
# Test the complete system
response = run_agent(
    prompt="""
    I want to buy 2 laptops and 3 mice for customer 12345.
    First check if these are available, then create the order.
    Finally, tell me the order status.
    """,
    tools=[
        search_products_by_name,
        check_product_availability,
        create_order_for_customer,
        get_order_status
    ],
    model="gpt-4o"
)

print("\n" + "="*60)
print("AGENT RESPONSE")
print("="*60)
print(response)

print("\n" + "="*60)
print("TOOL METRICS")
print("="*60)
for tool_name, metrics in metrics_registry.items():
    print(metrics.summary())

---

## Part 8: Exercises

### Exercise 1: Build a Banking Agent

Create tools for:
- `check_balance(account_id: int) -> str`
- `transfer_funds(from_account: int, to_account: int, amount: float) -> str`
- `get_transaction_history(account_id: int, days: int = 30) -> str`

Requirements:
- Validate account IDs exist
- Ensure sufficient balance before transfer
- Add error handling for all edge cases
- Include rate limiting (max 10 transfers per minute)
- Add metrics tracking

In [ ]:
# Your solution here


### Exercise 2: Debug This Tool

This tool has multiple issues. Find and fix them all.

In [ ]:
@tool
def buggy_booking_tool(date: str, time: str, guests: int) -> str:
    """Book a restaurant table."""
    # Issues:
    # 1. No date validation
    # 2. No time validation  
    # 3. No guest count validation
    # 4. No availability check
    # 5. No error handling
    return f"Booked for {date} at {time} for {guests} guests"

# Fix the tool above

### Exercise 3: Optimize Parallel Execution

Refactor this agent to use parallel function calling where possible.

In [ ]:
# Current: Sequential execution (slow)
@tool
def get_user_profile(user_id: int) -> str:
    time.sleep(1)
    return f"Profile for user {user_id}"

@tool
def get_user_orders(user_id: int) -> str:
    time.sleep(1)
    return f"Orders for user {user_id}"

@tool
def get_user_reviews(user_id: int) -> str:
    time.sleep(1)
    return f"Reviews by user {user_id}"

# Task: Make the agent call all three tools in parallel
# Hint: Adjust the prompt to encourage parallel execution

---

## Summary

You've learned:

1. **Schema Design** - Complex types, enums, Pydantic validation
2. **Error Handling** - Custom exceptions, retry logic, graceful failures
3. **Parallel Execution** - When and how to call functions in parallel
4. **Debugging** - Logging, metrics, validation testing
5. **Production Patterns** - Rate limiting, caching, observability
6. **Advanced Techniques** - Streaming, composition, context awareness

### Best Practices Checklist

- [ ] Use `Literal` types for fixed choices
- [ ] Provide detailed parameter descriptions
- [ ] Validate inputs with Pydantic
- [ ] Return structured JSON responses
- [ ] Add error handling for all edge cases
- [ ] Include retry logic for external APIs
- [ ] Cache expensive operations
- [ ] Rate limit API calls
- [ ] Track metrics (calls, duration, errors)
- [ ] Test tools independently before integration

---

## Next Steps

- **AG04**: Memory & State Management (build stateful agents)
- **AG06**: Reasoning & Planning (chain tools for complex tasks)
- **AG06.5**: Streaming Agents (real-time responses)

---

## Resources

- OpenAI Function Calling Guide: https://platform.openai.com/docs/guides/function-calling
- Pydantic Documentation: https://docs.pydantic.dev/
- Tool Design Patterns: See `documentation/tool-design-patterns.md`

---

*Estimated completion time: 8 hours*  
*Questions? See `AG03.5-SOLUTIONS.md` for exercise solutions*